# Lab 4 — Why It Matters for Agents: Do You Even Need a Model?

**AIEWF 2026 Workshop — Vector → Hybrid → Do You Even Need a Model?**

This notebook wires the Lab 3 hybrid retriever to an LLM synthesis call.

The thesis: **retrieval quality — not model quality — determines answer quality.**

---

## Requirements
```
# requirements.txt
elasticsearch>=8.17.0
requests
```

Environment variables needed (pre-configured in Instruqt sandbox):
- `ES_ENDPOINT` — your Elastic Serverless project URL
- `ES_API_KEY` — your Elastic API key

**No separate LLM key needed.** LLM synthesis uses the Elastic Inference API — your
ES credentials cover both search and LLM calls.


In [ ]:
# Cell 1 — Install and Import

# Uncomment to install if needed:
# !pip install elasticsearch requests

import os
import json
import requests
from elasticsearch import Elasticsearch

print('Imports OK')


In [ ]:
# Cell 2 — Connect to Elasticsearch
# Uses the same Serverless project as Labs 1-3.

ES_ENDPOINT = os.environ.get('ES_ENDPOINT')
ES_API_KEY = os.environ.get('ES_API_KEY')

if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        'Set ES_ENDPOINT and ES_API_KEY environment variables.\n'
        'In Instruqt: these are pre-configured in the sandbox environment.\n'
        'Running locally: export ES_ENDPOINT=https://... ES_API_KEY=...'
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

info = es.info()
print(f'Connected to Elasticsearch {info["version"]["number"]}')

# Verify corpus is indexed
count = es.count(index='aiewf-workshop-docs')['count']
print(f'Index has {count} documents')

In [ ]:
# Cell 3 — The Lab 3 Hybrid Retriever as a Python Function
#
# Same RRF retriever DSL from Lab 3 Dev Console, now as Python.
# ES client's search() takes the same body dict you'd paste into Dev Console.

INDEX_NAME = 'aiewf-workshop-docs'

def hybrid_search(query: str, size: int = 5) -> list[dict]:
    """
    Run a hybrid RRF search combining BM25 + semantic.
    
    Uses the same RRF retriever built in Lab 3.
    Returns a list of dicts: {id, title, url, body (truncated), score}
    """
    response = es.search(
        index=INDEX_NAME,
        body={
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": query,
                                        "fields": ["title^3", "body"],
                                        "type": "best_fields"
                                    }
                                }
                            }
                        },
                        {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "body_semantic",
                                        "query": query
                                    }
                                }
                            }
                        }
                    ],
                    "rank_constant": 60,
                    "rank_window_size": 100
                }
            },
            "size": size,
            "_source": ["id", "title", "url", "body"]
        }
    )
    
    results = []
    for hit in response['hits']['hits']:
        src = hit['_source']
        results.append({
            'id': src['id'],
            'title': src['title'],
            'url': src.get('url', ''),
            'body': src['body'][:800],  # truncate for context window management
            'score': hit['_score']
        })
    return results


# Quick test
results = hybrid_search("notify me when something goes wrong", size=3)
print(f'Hybrid search returned {len(results)} results:')
for r in results:
    print(f'  [{r["id"]}] {r["title"]} (score: {r["score"]:.4f})')

In [ ]:
# Cell 4 — LLM Synthesis Function
#
# Uses the Elastic Inference API — no separate LLM key needed.
# Your ES_API_KEY covers both search and LLM inference.
# We use Claude Haiku 4.5 (fast, cheap) — even it synthesizes a clear answer
# when context is correct (Cell 5), and fails when context is wrong (Cell 6).
#
# NOTE: we use the `completion` task type (not `chat_completion`). On Serverless the
# chat_completion task type only supports the streaming (_stream) API; the `completion`
# task type returns a single JSON response, which is simpler for a notebook.

INFERENCE_ID = '.anthropic-claude-4.5-haiku-completion'

SYSTEM_PROMPT = (
    'You are a helpful Elasticsearch documentation assistant. '
    'Answer the question using ONLY the provided documentation context. '
    'If the context does not contain enough information, say so clearly.'
)


def synthesize(context_docs: list[dict], question: str, inference_id: str = INFERENCE_ID) -> str:
    context_parts = []
    for i, doc in enumerate(context_docs, 1):
        context_parts.append(f"[Document {i}: {doc['title']}]\n{doc['body']}\n")
    context_text = '\n---\n'.join(context_parts)

    # The `completion` task type takes a single `input` string, so we fold the
    # system instructions, context, and question into one prompt.
    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context_text}\n\nQuestion: {question}"

    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/completion/{inference_id}",
        headers={
            'Authorization': f'ApiKey {ES_API_KEY}',
            'Content-Type': 'application/json',
        },
        json={'input': prompt},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()['completion'][0]['result']


print(f'synthesize() defined → Elastic Inference API ({INFERENCE_ID})')
print('No ANTHROPIC_API_KEY needed — ES credentials cover LLM inference too.')

## Cell 5 — Good Context Demo (PRE-BAKED)

This cell uses a **hardcoded question** and **pre-retrieved good context** — the kind of docs the hybrid retriever surfaces for "how do I get notified when something goes wrong" (the Watcher alerting page — doc-049 — plus the cluster-health and cat-API docs that an alert would actually watch).

**Run it as-is. Do not modify the context.**

The context is pre-baked so this demo is deterministic — the answer below was generated by the same model (Haiku 4.5) on exactly this context before the workshop.

### Why this works:
The hybrid retriever delivers docs that are genuinely *about* alerting. The Watcher page explains triggers, conditions, and actions; the cluster-health page describes the red/yellow signal a watch would fire on. The model synthesizes them into a concrete, correct answer. Even cheap Haiku does this well because the context is good.

In [ ]:
# Cell 5 — GOOD CONTEXT DEMO (pre-baked)
# DO NOT MODIFY — context is hardcoded for reproducibility

DEMO_QUESTION = "How do I get notified when something goes wrong in my cluster?"

# Pre-retrieved good context from the hybrid retriever
# (Real excerpts from the workshop corpus — verified to produce a good answer)
GOOD_CONTEXT = [
    {
        "id": "doc-049",
        "title": "Elasticsearch Watcher alerting",
        "url": "https://www.elastic.co/docs/explore-analyze/alerting/watcher",
        "body": (
            "Watcher is Elasticsearch's built-in alerting and notification system. Watches monitor "
            "conditions in your data and trigger actions when thresholds are met. A watch consists of "
            "a trigger (when it runs, typically a schedule), an input (what data to load, such as a "
            "search or an HTTP request to /_cluster/health), a condition (logic that decides whether "
            "to act, for example cluster status equals red), and actions (what to do when the condition "
            "is true: email, webhook, Slack, or index). Kibana Alerting (Rules and Connectors) is the "
            "modern alternative with a UI and more rule types; Watcher remains available for API-driven "
            "alert management."
        )
    },
    {
        "id": "doc-021",
        "title": "Red or yellow cluster health status troubleshooting",
        "url": "https://www.elastic.co/docs/troubleshoot/elasticsearch/red-yellow-cluster-status",
        "body": (
            "A red or yellow cluster health status indicates one or more shards are not assigned to a "
            "node. Red means unassigned primary shards (searches and indexing may fail); yellow means "
            "some replica shards are unassigned. Check GET /_cluster/health — a healthy cluster shows "
            "status green and unassigned_shards 0. A change to red or yellow is exactly the kind of "
            "condition an alert should watch for."
        )
    },
    {
        "id": "doc-043",
        "title": "Elasticsearch cat APIs overview",
        "url": "https://www.elastic.co/docs/reference/elasticsearch/rest-apis/compact-aliases",
        "body": (
            "The cat APIs return compact, human-readable operational views. GET /_cat/health shows "
            "cluster status; GET /_cat/nodes shows per-node CPU, heap, and load; GET /_cat/shards shows "
            "shard allocation and unassigned reasons. These endpoints are commonly polled by monitoring "
            "tools or used as a watch input to detect problems early."
        )
    }
]

print(f'Question: {DEMO_QUESTION}')
print(f'Context: {len(GOOD_CONTEXT)} documents (pre-baked good context)')
print('\n--- Answer (pre-generated by Haiku 4.5 on this exact context) ---\n')

GOOD_ANSWER = "Based on the provided documentation, here are the main ways to get notified when something goes wrong in your cluster:\n\n## Using Watcher (API-driven alerting)\n\nElasticsearch's built-in **Watcher** alerting system is designed for this purpose. A watch monitors conditions in your data and triggers actions when thresholds are met. A typical watch consists of:\n\n1. **Trigger** — defines when it runs (typically on a schedule)\n2. **Input** — loads data to check, such as a search or a request to `/_cluster/health`\n3. **Condition** — logic that decides whether to act (for example, cluster status equals red)\n4. **Actions** — what to do when the condition is true, including:\n   - Email\n   - Webhook\n   - Slack\n   - Index a document\n\n## Example Use Case\n\nFor cluster health monitoring specifically, you could create a watch that:\n- Uses `GET /_cluster/health` as input to check cluster status\n- Monitors for red or yellow status (which indicate unassigned shards)\n- Sends notifications via email, Slack, or webhook when problems are detected\n\n## Modern Alternative\n\nThe documentation also mentions that **Kibana Alerting (Rules and Connectors)** is a modern alternative with a UI and more rule types, though Watcher remains available for API-driven alert management."
print(GOOD_ANSWER)

## Cell 6 — Bad Context Demo (PRE-BAKED)

**Same question. Same model. Same prompt. Different context.**

We replaced the retrieved docs with irrelevant documents about ILM lifecycle policies and snapshot configuration.

### The model didn't get dumber. The retrieval got worse.

This is the core thesis of the workshop. The LLM is a synthesis engine — it works with what you give it. Bad retrieval → bad context → bad answer. The model is not the failure point.

In [ ]:
# Cell 6 — BAD CONTEXT DEMO (pre-baked)
# Same question, same model, same prompt — deliberately wrong context
# DO NOT MODIFY — context is hardcoded to produce a bad answer reproducibly

# Same question as Cell 5
# DEMO_QUESTION = "How do I get notified when something goes wrong in my cluster?"

# Deliberately wrong context: ILM, snapshots, ingest pipelines — nothing about alerting
BAD_CONTEXT = [
    {
        "id": "doc-017",
        "title": "Index lifecycle management overview",
        "url": "https://www.elastic.co/docs/manage-data/lifecycle/index-lifecycle-management",
        "body": (
            "Index Lifecycle Management (ILM) automates index management through hot, warm, cold, "
            "frozen, and delete phases. Define a policy and apply it via an index template; use the "
            "rollover action to create new indices when the current one reaches a size or age threshold."
        )
    },
    {
        "id": "doc-037",
        "title": "Snapshot and restore overview",
        "url": "https://www.elastic.co/docs/deploy-manage/tools/snapshot-and-restore",
        "body": (
            "Snapshots back up cluster data to a repository (S3, GCS, Azure, or shared filesystem). "
            "Create a repository, take a snapshot, and restore from it. Snapshot lifecycle management "
            "(SLM) automates scheduled snapshots."
        )
    },
    {
        "id": "doc-019",
        "title": "Ingest pipelines in Elasticsearch",
        "url": "https://www.elastic.co/docs/manage-data/ingest/transform-enrich/ingest-pipelines",
        "body": (
            "Ingest pipelines transform documents before indexing using processors such as set, grok, "
            "date, rename, and script. Create a pipeline with PUT /_ingest/pipeline and apply it via "
            "the index default_pipeline setting or per request."
        )
    }
]

print(f'Question: {DEMO_QUESTION}')
print(f'Context: {len(BAD_CONTEXT)} documents (WRONG context — ILM, snapshots, pipelines)')
print('\n--- Answer (pre-generated by Haiku 4.5 on this exact context) ---\n')

BAD_ANSWER = "I don't have enough information in the provided documentation to answer your question about cluster notifications or alerts.\n\nThe documentation context covers:\n- Index Lifecycle Management (ILM)\n- Snapshots and restore\n- Ingest pipelines\n\nNone of these sections address cluster monitoring, alerting, or notification mechanisms. To answer your question properly, I would need documentation covering topics like Elasticsearch alerting, monitoring, or watcher functionality.\n\nI recommend checking the Elasticsearch documentation for sections on \"Alerting,\" \"Watcher,\" \"Stack Monitoring,\" or \"Notifications\" for this information."
print(BAD_ANSWER)

print('\n' + '='*60)
print('THE LESSON: The model did not get dumber. The retrieval got worse.')
print('Retrieval quality determines answer quality.')

In [ ]:
# Cell 7 — Full Pipeline End-to-End  [INSTRUCTOR DEMO — runs on screen]
#
# hybrid_search() + synthesize() both use ES credentials — nothing else needed.

def ask(question: str, inference_id: str = INFERENCE_ID) -> str:
    """Full RAG pipeline: hybrid retrieve → LLM synthesize via Elastic Inference API."""
    print(f'Retrieving context for: "{question}"')
    docs = hybrid_search(question, size=5)
    print(f'Retrieved {len(docs)} docs:')
    for d in docs:
        print(f'  [{d["id"]}] {d["title"]}')
    print('\nSynthesizing via Elastic Inference API...')
    return synthesize(docs, question, inference_id=inference_id)


answer = ask("notify me when something goes wrong")
print('\n--- Answer ---')
print(answer)

In [ ]:
# Cell 7b — Try your own questions  [TAKE-HOME]
# Requires Cell 7 to have run first (ask() must be defined).

YOUR_QUESTION = "how do I set up TLS for my cluster?"  # Change this

answer = ask(YOUR_QUESTION)
print('\n--- Answer ---')
print(answer)


## Cell 8 — Minimal Multi-Hop Agent Loop (Take-Home)

**This cell is take-home only** — skip it during the workshop.

Requires Cell 7 to have run (needs `synthesize()`, `hybrid_search()`, and `ask()`).

A minimal agent that decides when to retrieve again. Each hop uses the prior result.
The point: ES does the heavy lifting on every retrieval hop.

In [ ]:
# Cell 8 — Multi-Hop Agent Loop (TAKE-HOME — skip during workshop)
# The agent retrieves twice — second hop is informed by the first result.
# Uses the Elastic Inference API for the follow-up query decision — no extra keys.

def multi_hop_agent(initial_question: str, max_hops: int = 2) -> str:
    """
    Minimal agent loop: retrieve → synthesize → decide to retrieve again.

    Hop 1: retrieve docs for the initial question
    Hop 2: if the answer suggests a follow-up, form a refined query and retrieve again
    """
    print(f'Starting multi-hop agent for: "{initial_question}"')
    all_context = []
    current_question = initial_question

    for hop in range(max_hops):
        print(f'\n--- Hop {hop + 1} ---')
        print(f'Query: "{current_question}"')

        docs = hybrid_search(current_question, size=3)
        print(f'Retrieved: {[d["title"] for d in docs]}')
        all_context.extend(docs)

        if hop == max_hops - 1:
            break

        # Ask the model whether a follow-up search would help. We reuse the same
        # `completion` task type as synthesize() — fold the instruction into `input`.
        context_text = '\n'.join([f'[{d["title"]}]: {d["body"][:300]}' for d in docs])
        decision_prompt = (
            'You are a search agent. Given a question and initial search results, decide if a '
            'follow-up search query would help answer the question better. If yes, output ONLY the '
            'follow-up query text. If no follow-up is needed, output: DONE\n\n'
            f'Question: {initial_question}\n\nInitial results:\n{context_text}\n\nFollow-up query or DONE?'
        )
        resp = requests.post(
            f"{ES_ENDPOINT}/_inference/completion/{INFERENCE_ID}",
            headers={'Authorization': f'ApiKey {ES_API_KEY}', 'Content-Type': 'application/json'},
            json={'input': decision_prompt},
            timeout=60,
        )
        resp.raise_for_status()
        followup = resp.json()['completion'][0]['result'].strip()

        if followup == 'DONE' or not followup:
            print('Agent decided: no follow-up needed')
            break
        else:
            print(f'Agent follow-up query: "{followup}"')
            current_question = followup

    seen = set()
    unique_context = []
    for doc in all_context:
        if doc['id'] not in seen:
            seen.add(doc['id'])
            unique_context.append(doc)

    print(f'\n--- Final synthesis ({len(unique_context)} unique docs) ---')
    return synthesize(unique_context, initial_question)


result = multi_hop_agent("authentication issues after upgrading to 9.0")
print('\n=== Final Answer ===')
print(result)